### Question 1: [Index] S&P 500 Stocks Added to the Index
Which year had the highest number of additions (starting from 2020)?



In [3]:
import pandas as pd
from io import StringIO

In [2]:
import requests

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)

In [ ]:
# 1. Create a DataFrame with company tickers, names, and the year they were added.
df_sp500 = pd.read_html(StringIO(response.text))[0]

# Extract the year from the addition date
df_sp500['Year added'] = pd.to_datetime(df_sp500['Date added'], errors='coerce').dt.year

# Filter the DataFrame to keep only the wanted columns
df_sp500 = df_sp500[['Symbol', 'Security', 'Year added']]

df_sp500.head()

,Symbol,Security,Year added
0,MMM,3M,1957
1,AOS,A. O. Smith,2017
2,ABT,Abbott Laboratories,1957
3,ABBV,AbbVie,2012
4,ACN,Accenture,2011


In [11]:
# Calculate the number of stocks added each year
stocks_added_per_year = df_sp500.groupby('Year added').size().reset_index(name='Count')
stocks_added_per_year.T

,0,1,2,3,4,5,6,7,8,9,...,48,49,50,51,52,53,54,55,56,57
Year added,1957,1964,1965,1969,1970,1972,1973,1974,1975,1976,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Count,52,1,2,2,2,2,2,1,2,11,...,22,13,21,10,10,15,15,16,18,13


In [15]:
# Which (full) year had the highest number of additions, starting from 2020?
recent_additions = stocks_added_per_year[stocks_added_per_year['Year added'] >= 2020]
recent_additions = recent_additions.sort_values(by='Count', ascending=False)
recent_additions.head()

,Year added,Count
56,2025,18
55,2024,16
53,2022,15
54,2023,15
57,2026,13


Answer for the 1st question: 2025

In [16]:
# Additional: How many current S&P 500 stocks have been in the index for more than 20 years?
current_year = 2026
stocks_in_index_for_more_than_20_years = df_sp500[df_sp500['Year added'] < current_year - 20]
print(f"Number of S&P 500 stocks in the index for more than 20 years: {stocks_in_index_for_more_than_20_years.shape[0]}")

Number of S&P 500 stocks in the index for more than 20 years: 218


### Question 2. [Macro] Indexes YTD (as of 21 August 2026)
How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?

In [17]:
import yfinance as yf

indices = {
    'S&P 500': '^GSPC',
    'Shanghai': '000001.SS',
    'Hang Seng': '^HSI',
    'ASX 200': '^AXJO',
    'Nifty 50': '^NSEI',
    'TSX': '^GSPTSE',
    'DAX': '^GDAXI',
    'FTSE 100': '^FTSE',
    'Nikkei 225': '^N225',
    'IPC Mexico': '^MXX',
    'Ibovespa': '^BVSP'
}

In [21]:
results = []

for name, ticker in indices.items():
    df = yf.download(ticker, start='2026-01-01', end='2026-08-21', progress=False)
    
    if not df.empty:
        close_series = df['Close'].dropna().values.flatten()
        
        # first and last closing prices
        first_close = float(close_series[0])
        last_close = float(close_series[-1])
        
        ytd_return = (last_close / first_close) -1
        
        results.append({
            'Index Name': name,
            'Ticker': ticker,
            'YTD Return (%)': ytd_return * 100
        })
        
# Create a DataFrame from the results and sort by YTD Return
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='YTD Return (%)', ascending=False).reset_index(drop=True)

df_results

,Index Name,Ticker,YTD Return (%)
0,Nikkei 225,^N225,27.750745
1,TSX,^GSPTSE,14.057466
2,S&P 500,^GSPC,11.412019
3,FTSE 100,^FTSE,8.010176
4,DAX,^GDAXI,5.883203
5,Ibovespa,^BVSP,4.601997
6,ASX 200,^AXJO,4.078920
7,IPC Mexico,^MXX,0.324972
8,Hang Seng,^HSI,-2.429832
9,Shanghai,000001.SS,-2.974985


In [22]:
# Get the S&P 500 return
sp500_return = df_results.loc[df_results['Ticker'] == '^GSPC', 'YTD Return (%)'].values[0]

sp500_return


np.float64(11.412019253393835)

In [23]:
# find the better performing index compared to S&P 500
better_performing_indices = df_results[df_results['YTD Return (%)'] > sp500_return]
better_performing_indices

,Index Name,Ticker,YTD Return (%)
0,Nikkei 225,^N225,27.750745
1,TSX,^GSPTSE,14.057466


Answer for the 2nd question is: 2

In [27]:
# Additional: How many of these indexes have better returns than the S&P 500 over 3, 5, and 10 year periods? Do you see the same trend?

end_date = '2026-08-21'
periods = {"3 Year" : 3, "5 Year" : 5, "10 Year" : 10}

results = []

for name, ticker in indices.items():
    row = {'Index Name': name, 'Ticker': ticker}
    for period_name, years in periods.items():
        start_year = 2026 - years
        start_date = f'{start_year}-08-21'
        
        df = yf.download(ticker, start=start_date, end=end_date, progress=False)
        if not df.empty:
            close_series = df['Close'].dropna().values.flatten()
            first_close = float(close_series[0])
            last_close = float(close_series[-1])
            return_value = (last_close / first_close) - 1
            row[period_name] = return_value * 100
    results.append(row)

df_results = pd.DataFrame(results)
df_results

,Index Name,Ticker,3 Year,5 Year,10 Year
0,S&P 500,^GSPC,73.671808,70.579514,250.087991
1,Shanghai,000001.SS,26.212293,12.268482,26.546768
2,Hang Seng,^HSI,45.821192,2.345321,11.742719
3,ASX 200,^AXJO,27.662143,21.280657,64.707796
4,Nifty 50,^NSEI,24.947664,46.891305,180.813852
5,TSX,^GSPTSE,83.803798,77.588828,146.575161
6,DAX,^GDAXI,66.522927,63.901995,147.590751
7,FTSE 100,^FTSE,48.091715,51.191450,57.402068
8,Nikkei 225,^N225,109.774894,140.838767,298.939832
9,IPC Mexico,^MXX,21.163791,23.619720,33.247441


In [29]:
# Sort for each period
for period in periods.keys():
    df_results = df_results.sort_values(by=period, ascending=False).reset_index(drop=True)
    print(f"Top performers for {period}:")
    print(df_results[['Index Name', period]].head())
    print("\n")

Top performers for 3 Year:
   Index Name      3 Year
0  Nikkei 225  109.774894
1         TSX   83.803798
2     S&P 500   73.671808
3         DAX   66.522927
4    FTSE 100   48.091715


Top performers for 5 Year:
   Index Name      5 Year
0  Nikkei 225  140.838767
1         TSX   77.588828
2     S&P 500   70.579514
3         DAX   63.901995
4    FTSE 100   51.191450


Top performers for 10 Year:
   Index Name     10 Year
0  Nikkei 225  298.939832
1     S&P 500  250.087991
2    Ibovespa  190.626677
3    Nifty 50  180.813852
4         DAX  147.590751




### Question 3. [Index] S&P 500 Market Corrections Analysis
Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.



In [2]:
import yfinance as yf
import pandas as pd
import numpy as np


In [3]:
# Download S&P 500 historical data (period from 1950 to present) using yfinance (daily stats)
sp500 = yf.download('^GSPC', start='1950-01-01', progress=False)

sp500.head()

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
1950-01-03,16.66,16.66,16.66,16.66,1260000
1950-01-04,16.85,16.85,16.85,16.85,1890000
1950-01-05,16.93,16.93,16.93,16.93,2550000
1950-01-06,16.98,16.98,16.98,16.98,2010000
1950-01-09,17.08,17.08,17.08,17.08,2520000


In [4]:
# Get the close price as Series
sp500_close = sp500['Close'].squeeze().dropna()

# Identify all-time high closing price points (where price exceeds all previous days' prices)
cummax = sp500_close.cummax()
ath_mask = cummax.diff() > 0

# Include initial all-time high (first closing price) as well
ath_mask.iloc[0] = True
ath_dates = sp500_close[ath_mask].index

corrections = []

# For each pair of consecutive all-time highs, find the minimum price in between
for i in range(len(ath_dates) - 1):
    peak_date = ath_dates[i]
    next_ath_date = ath_dates[i + 1]
    
    # Get the period between the two all-time highs
    period = sp500_close[peak_date:next_ath_date]
    
    # Trading days between the two all-time highs
    if len(period) > 2:
        high_price = float(period.iloc[0])
        low_price = float(period.min())
        trough_date = period.idxmin()
        
        # Calculate the drawdown percentage
        drawdown = (high_price - low_price) / high_price * 100
        
        # Filter for corrections with at least 5% drawdown
        if drawdown >= 5.0:
            #Calculate the duration of the correction in trading days
            duration_days = (trough_date - peak_date).days
            
            corrections.append({
                'Peak Date': peak_date.strftime('%Y-%m-%d'),
                'Trough Date': trough_date.strftime('%Y-%m-%d'),
                'Drawdown (%)': drawdown,
                'Duration (days)': duration_days
            })

df_corrections = pd.DataFrame(corrections)

In [5]:
# Compare top 10 largest corrections with the hint
print("Top 10 largest corrections:")
print(df_corrections.sort_values(by='Drawdown (%)', ascending=False).head(10))

Top 10 largest corrections:
     Peak Date Trough Date  Drawdown (%)  Duration (days)
56  2007-10-09  2009-03-09     56.775388              517
54  2000-03-24  2002-10-09     49.146948              929
24  1973-01-11  1974-10-03     48.203593              630
22  1968-11-29  1970-05-26     36.061641              543
65  2020-02-19  2020-03-23     33.924960               33
35  1987-08-25  1987-12-04     33.509515              101
15  1961-12-12  1962-06-26     27.973568              196
27  1980-11-28  1982-08-12     27.113582              622
68  2022-01-03  2022-10-12     25.425097              282
18  1966-02-09  1966-10-07     22.177335              240


In [7]:
# Determine the 25th, 50th (median), and 75th percentiles for correction durations and drawdowns
percentiles = [0.25, 0.5, 0.75]

print("\n---Drawdown Percentiles---")
print(df_corrections['Drawdown (%)'].quantile(percentiles).round(2))

print("\n---Duration Percentiles---")
print(df_corrections['Duration (days)'].quantile(percentiles).round(2))


---Drawdown Percentiles---
0.25     6.23
0.50     7.99
0.75    14.02
Name: Drawdown (%), dtype: float64

---Duration Percentiles---
0.25    22.00
0.50    40.50
0.75    86.25
Name: Duration (days), dtype: float64


### Question 4. [Stocks] Earnings Surprise Analysis for Amazon (AMZN)
Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.

In [18]:
# Load earnings data for the last few years using the yfinance method get_earnings_dates():
import yfinance as yf
import pandas as pd

ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
earnings_df = ticker_obj.get_earnings_dates(limit=25)
price_df = ticker_obj.history(period='5y')

price_df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2021-09-14 00:00:00-04:00,173.777496,174.340500,171.885498,172.500000,38738000,0.0,0.0
2021-09-15 00:00:00-04:00,172.126007,174.270996,170.100494,173.789505,59150000,0.0,0.0
2021-09-16 00:00:00-04:00,172.998001,174.627502,172.307007,174.412003,51672000,0.0,0.0
2021-09-17 00:00:00-04:00,174.420502,174.870499,172.606506,173.126007,92332000,0.0,0.0
2021-09-20 00:00:00-04:00,169.800003,170.949997,165.250504,167.786499,93382000,0.0,0.0


In [19]:
# Calculate 2-day percentage changes for all historical dates: for each sequence of 3 consecutive trading days (Day 1, Day 2, Day 3), compute the return as Close_Day3 / Close_Day1 - 1. (Assume Day 2 may correspond to the earnings announcement.)
price_df['2d_return'] = price_df['Close'].shift(-1) / price_df['Close'].shift(1) - 1

In [20]:
# Normalize the earnings dates to match
earnings_df = earnings_df.dropna(subset=['Surprise(%)']).copy()
earnings_df.index = pd.to_datetime(earnings_df.index).tz_localize(None).normalize()

price_df.index = pd.to_datetime(price_df.index).tz_localize(None).normalize()

# Merge
merged_df = earnings_df.join(price_df[['2d_return']], how='inner')

In [21]:
#Filter for positive earnings surprises and calculate the median 2-day return. Then calculate the correlation between
# the 2-day stock return and the earnings surprise magnitude.

positive_surprises = merged_df[merged_df['Surprise(%)'] > 0].copy()

# Calculate the median 2-day return 
median_2d_return = positive_surprises['2d_return'].median()

# Compute correlation between surprise magnitude and 2-day return
correlation = positive_surprises['Surprise(%)'].corr(positive_surprises['2d_return'])

print(f"Median 2-day return for positive earnings surprises: {median_2d_return:.4f}")
print(f"\nCorrelation:")
print(correlation)

Median 2-day return for positive earnings surprises: 0.0238

Correlation:
0.31923036721072084
